# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("ok")
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )



# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [5]:
nutrition_agent = Agent(
    name="nutrition assistant", 
    instructions="""You're a helpful assistant giving out nutrition advice
    You give consise answers
    """
)

Let's execute the Agent:

In [6]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="nutrition assistant", ...)
- Final output (str):
    Bananas are generally healthy. Key benefits:
    - Rich in potassium, vitamin B6, vitamin C, and dietary fiber
    - Provide quick energy from natural sugars
    - Unripe bananas have more resistant starch (great for digestion); ripe bananas are softer and sweeter
    
    Considerations:
    - Moderate sugar content; portion if managing blood sugar
    - Good as part of a balanced snack (with protein/fat)
    
    Bottom line: a nutritious, convenient fruit when eaten as part of a varied diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [7]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are generally healthy.

- Pros: good source of potassium, vitamin C, vitamin B6, and dietary fiber; low in fat; convenient quick energy.
- Cons: contain natural sugars and carbs; portion matters for those watching carbs or blood sugar.
- Tips: one medium banana (~105 kcal) fits most diets; greener bananas have more resistant starch, riper ones sweeter.
- Bottom line: they’re a nutritious, versatile fruit—enjoy in moderation as part of a balanced diet.

_Good Job!_